# 09 — Moteur A : expériences (validation uniquement)
Protocole fixé avant exécution : 4 variantes, évaluation sur la validation seulement,
budget d'alertes 0,179 %. Critère de retenue : PR-AUC validation ≥ 2 × référence (≥ 0,045)
ET rappel au budget amélioré. Seule une variante retenue sera évaluée (une fois) sur les tests.

In [ ]:
import gc
import sys
import time
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import average_precision_score

PROJET = Path("/mnt/c/Users/Salah/Desktop/SecuMon-IA")
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists()), PROJET)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from secumon.ml.models import anomaly, dataset

GOLD = ROOT / "data" / "processed" / "gold" / "gold.parquet"
TMP_LINUX = Path.home() / ".cache" / "secumon-duckdb-tmp"
TMP_LINUX.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f"SET temp_directory = '{TMP_LINUX.as_posix()}'")
con.execute("SET memory_limit = '1GB'")
con.execute("SET threads = 2")
con.execute("SET enable_progress_bar = false")

TAUX_ALERTE = 3042 / 1701371          # budget d'alertes du moteur B sur la validation
print("GOLD existe :", GOLD.exists())

In [ ]:
t0 = time.perf_counter()
X_tr = anomaly.anomaly_matrix(dataset.load_sample(con, GOLD, "train", every=20))

morceaux_X, morceaux_y = [], []
for part in dataset.iter_split(con, GOLD, "validation", n_parts=4):
    morceaux_X.append(anomaly.anomaly_matrix(part))
    morceaux_y.append(part["is_fraud"].to_numpy(dtype=bool))
    del part
    gc.collect()
X_val = pd.concat(morceaux_X, ignore_index=True)
y_val = np.concatenate(morceaux_y)
del morceaux_X, morceaux_y
gc.collect()

print(f"Preparation : {time.perf_counter() - t0:.0f} s")
print(f"Train : {X_tr.shape} | Validation : {X_val.shape}, {int(y_val.sum()):,} fraudes")

In [ ]:
GEO_NOUVEAUTE = ["online", "country_new", "merchant_new", "mcc_new",
                 "dist_home_km", "dist_prev_km", "speed_kmh", "amt_ratio_avg_90d"]
VELOCITE_MONTANT = ["amount", "amt_ratio_avg_90d", "amt_zscore_90d", "n_tx_1h", "n_tx_24h",
                    "amt_sum_24h", "n_err_24h", "secs_since_prev"]

VARIANTES = {
    "V1 reference (16 features)":        (anomaly.ANOMALY_COLUMNS, "auto"),
    "V2 nouveaute + geographie":         (GEO_NOUVEAUTE, "auto"),
    "V3 velocite + montant":             (VELOCITE_MONTANT, "auto"),
    "V4 reference, max_samples 4096":    (anomaly.ANOMALY_COLUMNS, 4096),
}

resultats = []
for nom, (cols, max_samples) in VARIANTES.items():
    t0 = time.perf_counter()
    model = IsolationForest(n_estimators=200, max_samples=max_samples,
                            random_state=42, n_jobs=4).fit(X_tr[cols])
    score = -model.score_samples(X_val[cols])
    seuil = np.quantile(score, 1 - TAUX_ALERTE)
    alerte = score >= seuil
    tp = int((alerte & y_val).sum())
    resultats.append({
        "variante": nom,
        "n_features": len(cols),
        "pr_auc": round(average_precision_score(y_val, score), 4),
        "rappel": round(tp / y_val.sum(), 4),
        "precision": round(tp / alerte.sum(), 4),
        "duree_s": round(time.perf_counter() - t0),
    })
    print(f"{nom} : PR-AUC {resultats[-1]['pr_auc']}, rappel {resultats[-1]['rappel']}")
    del model, score
    gc.collect()

res = pd.DataFrame(resultats)
ref = res.loc[0, "pr_auc"]
res["x_reference"] = (res["pr_auc"] / ref).round(2)
res["retenue"] = (res["pr_auc"] >= 2 * ref) & (res["rappel"] > res.loc[0, "rappel"])
print(res.to_string(index=False))

In [ ]:
# 2e et DERNIERE manche, declaree apres la 1re : V5 = V2 (nouveaute + geographie) + max_samples 4096.
# Meme critere qu'en 1re manche : PR-AUC >= 2 x reference ET rappel > reference.
t0 = time.perf_counter()
model = IsolationForest(n_estimators=200, max_samples=4096,
                        random_state=42, n_jobs=4).fit(X_tr[GEO_NOUVEAUTE])
score = -model.score_samples(X_val[GEO_NOUVEAUTE])
seuil = np.quantile(score, 1 - TAUX_ALERTE)
alerte = score >= seuil
tp = int((alerte & y_val).sum())

v5 = {
    "variante": "V5 nouveaute + geographie, max_samples 4096 (2e manche)",
    "n_features": len(GEO_NOUVEAUTE),
    "pr_auc": round(average_precision_score(y_val, score), 4),
    "rappel": round(tp / y_val.sum(), 4),
    "precision": round(tp / alerte.sum(), 4),
    "duree_s": round(time.perf_counter() - t0),
}
v5["x_reference"] = round(v5["pr_auc"] / ref, 2)
v5["retenue"] = (v5["pr_auc"] >= 2 * ref) and (v5["rappel"] > res.loc[0, "rappel"])

res_final = pd.concat([res, pd.DataFrame([v5])], ignore_index=True)
print(res_final.to_string(index=False))

In [ ]:
import importlib
import json

import joblib

importlib.reload(anomaly)
MODELS_A = ROOT / "data" / "processed" / "models" / "engine_a"

model_v2 = anomaly.fit_isolation_forest(X_tr)          # reglages par defaut = V5
s_val = anomaly.anomaly_score(model_v2, X_val)
SEUIL_A2 = float(np.quantile(s_val, 1 - TAUX_ALERTE))
alerte = s_val >= SEUIL_A2
tp = int((alerte & y_val).sum())

print("Colonnes du modele :", list(model_v2.feature_names_in_))
print(f"PR-AUC validation  : {average_precision_score(y_val, s_val):.4f}  (attendu 0.0677)")
print(f"Rappel / precision : {tp / y_val.sum():.4f} / {tp / alerte.sum():.4f}  (attendu 0.1341 / 0.1446)")

joblib.dump(model_v2, MODELS_A / "isolation_forest_v2.joblib")
(MODELS_A / "meta_v2.json").write_text(json.dumps({
    "version": "v2", "threshold": SEUIL_A2, "alert_rate": TAUX_ALERTE,
    "columns": list(anomaly.SELECTED_COLUMNS), "max_samples": anomaly.DEFAULT_MAX_SAMPLES,
    "n_estimators": 200, "seed": 42, "train_sample_every": 20,
}, indent=2))
print("Modele v2 sauvegarde.")

del X_tr, X_val, y_val, s_val
gc.collect()

In [ ]:
t0 = time.perf_counter()
morceaux = []
for split in ["test_a", "test_b"]:
    for part in dataset.iter_split(con, GOLD, split, n_parts=12):
        morceaux.append(pd.DataFrame({
            "tx_id": part["tx_id"].to_numpy(),
            "score_a2": anomaly.anomaly_score(model_v2, anomaly.anomaly_matrix(part)),
        }))
        del part
        gc.collect()
scores_a2 = pd.concat(morceaux, ignore_index=True)
del morceaux
scores_a2.to_parquet(MODELS_A / "scores_tests_v2.parquet", index=False)

MODELS_B = ROOT / "data" / "processed" / "models" / "engine_b"
SEUIL_B = json.loads((MODELS_B / "meta.json").read_text())["threshold"]
scores = pd.read_parquet(MODELS_B / "scores_tests.parquet").merge(scores_a2, on="tx_id")
print(f"Scoring : {time.perf_counter() - t0:.0f} s | lignes : {len(scores):,}  (attendu 7,214,337)")

In [ ]:
def bilan(df):
    y = df["is_fraud"].to_numpy(dtype=bool)
    a = df["score_a2"].to_numpy() >= SEUIL_A2
    b = df["score"].to_numpy() >= SEUIL_B
    out = {"fraudes": int(y.sum()),
           "pr_auc_A2": round(average_precision_score(y, df["score_a2"]), 4) if 0 < y.sum() < len(y) else np.nan}
    for nom, alerte in [("A2", a), ("B", b), ("A2 ou B", a | b)]:
        tp, n_a = int((alerte & y).sum()), int(alerte.sum())
        out[f"rappel {nom}"] = round(tp / y.sum(), 4) if y.sum() else np.nan
        out[f"precision {nom}"] = round(tp / n_a, 4) if n_a else np.nan
        out[f"alertes {nom}"] = n_a
    out["manquees_B"] = int((y & ~b).sum())
    out["rattrapees_par_A2"] = int((y & ~b & a).sum())
    return pd.Series(out)

cols = ["is_fraud", "score", "score_a2"]
print("=== Global ===")
print(scores.groupby("split")[cols].apply(bilan).T.to_string())
print("\n=== Par zone (zones avec fraudes) ===")
par_zone = scores.groupby(["split", "zone"])[cols].apply(bilan)
print(par_zone[par_zone["fraudes"] > 0].T.to_string())

## Conclusion — moteur A v2 retenu

**Protocole** : 1re manche (V1–V4) puis 2e et dernière manche déclarée (V5), même critère fixé
avant exécution (PR-AUC validation ≥ 2 × référence ET rappel amélioré), tests jamais utilisés pour choisir.
V5 retenue : 8 features (nouveauté + géographie), max_samples 4096 → PR-AUC validation 0,0677 (× 3).
Le module reproduit exactement l'expérience (0,0677 / 0,1341 / 0,1446).

**Évaluation unique sur les tests**

| | V1 | V2 |
|---|---|---|
| Rappel test A / test B | 3,2 % / 9,5 % | 12,1 % / 37,1 % |
| Précision test A / test B | 4,3 % / 5,4 % | 15,6 % / 16,7 % |
| Fraudes italiennes manquées par B rattrapées | 114 / 2 801 | 587 / 2 801 |

**Où A2 est utile** : Italie (contexte nouveau) — précision 97,8 % ; bruit sur États-Unis (0,04 %) et en ligne (0,65 %).
**« A2 ou B »** : test B 41,1 % → 53,3 % de rappel, mais alertes × 4,1 (précision 66 % → 21 %) → OU simple rejeté.

**Enseignements** : le signal non supervisé utile est dans la nouveauté et la géographie ; la vélocité
seule n'est pas un bon détecteur d'anomalies (V3). Le moteur de décision doit combiner les scores avec
nuance (B : refus ; A2 : revue analyste), et non par un OU.